# Fully Bayesian SAAS GP

SAAS は、入力次元が比較的高く、そのうち実際に効いている次元が少数だと考えられる場合に有効です。この Notebook では single-task 版と multi-task 版の両方を扱います。Fully Bayesian SAAS は `make_mll()` ではなく NUTS で学習します。

> この例ではドキュメントとして実行しやすいように MCMC の設定を小さくしています。実解析では `warmup_steps` と `num_samples` を増やしてください。

In [ ]:
import torch
import matplotlib.pyplot as plt
from botorch.fit import fit_fully_bayesian_model_nuts
from robotorchan.models import SaasFullyBayesianSingleTaskGP, SaasFullyBayesianMultiTaskGP

torch.manual_seed(0)
dtype = torch.double

## 1. 高次元の合成データ

In [ ]:
n, d = 28, 12
train_X = torch.rand(n, d, dtype=dtype)
def f(X):
    return (torch.sin(2 * torch.pi * X[..., 0]) + 0.8 * (X[..., 2] - 0.5) ** 2 - 0.6 * X[..., 5]).unsqueeze(-1)
train_Y = f(train_X) + 0.03 * torch.randn(n, 1, dtype=dtype)
train_X.shape, train_Y.shape

## 2. Single-task SAAS

まず単一タスクの SAAS GP を構築します。`raw_train_X` / `raw_train_Y` はコンストラクタに渡した元データを保持します。`supports_mll=False` は仕様であり、学習には NUTS を使用します。

In [ ]:
model = SaasFullyBayesianSingleTaskGP(train_X, train_Y)
print(model.raw_train_X.shape, model.raw_train_Y.shape)
print('supports_mll =', model.supports_mll)

In [ ]:
fit_fully_bayesian_model_nuts(
    model,
    warmup_steps=32,
    num_samples=16,
    thinning=2,
    disable_progbar=True,
)

In [ ]:
x0 = torch.linspace(0, 1, 120, dtype=dtype)
X_test = torch.full((120, d), 0.5, dtype=dtype)
X_test[:, 0] = x0
with torch.no_grad():
    posterior = model.posterior(X_test)
mean = posterior.mean.squeeze(-1)
std = posterior.variance.sqrt().squeeze(-1)
plt.figure(figsize=(7, 4))
plt.plot(x0, mean)
plt.fill_between(x0, mean - 2 * std, mean + 2 * std, alpha=0.2)
plt.xlabel('x0')
plt.ylabel('posterior')
plt.title('SAAS posterior slice')
plt.show()

## 3. Multi-task SAAS

次に BoTorch の long-format task feature を使う multi-task SAAS を構築します。最後の列を `task_feature` として扱います。

In [ ]:
x_mt = torch.rand(18, d, dtype=dtype)
X0 = torch.cat([x_mt, torch.zeros(18, 1, dtype=dtype)], dim=-1)
X1 = torch.cat([x_mt, torch.ones(18, 1, dtype=dtype)], dim=-1)
Y0 = f(x_mt)
Y1 = 0.7 * f(x_mt) + 0.2
train_X_mt = torch.cat([X0, X1], dim=0)
train_Y_mt = torch.cat([Y0, Y1], dim=0)
mt_model = SaasFullyBayesianMultiTaskGP(train_X_mt, train_Y_mt, task_feature=-1)
print(mt_model.raw_train_X.shape, mt_model.raw_train_Y.shape)
print('supports_mll =', mt_model.supports_mll)

In [ ]:
fit_fully_bayesian_model_nuts(
    mt_model,
    warmup_steps=24,
    num_samples=12,
    thinning=2,
    disable_progbar=True,
)

## 4. 使い所と注意点

- 高次元で、重要な入力次元が少数だと考えられる場合に SAAS を検討します。
- 計算時間を重視する場合は通常 GP や MAP-SAAS 系も候補です。
- `supports_mll=False` は意図した仕様で、学習経路は NUTS です。
- MCMC は比較的重いため、高速な PR 用 CI では実行を省略するか、さらに軽い設定を使用してください。